# PEAKS spectral library reader

In [1]:
%reload_ext autoreload
%autoreload 2

`PEAKSLibraryReader` reads spectral libraries exported by PEAKS Studio and returns an `alphabase.spectral_library.flat.SpecLibFlat`.

PEAKS' format is one row per precursor (unlike e.g. Spectronaut's one-row-per-fragment format handled by `LibraryReaderBase`), with all fragment ions packed into a single `Peaks List` column and modifications reported as `<0-based position>-<name>-(<mass>)` tokens, e.g. `10-Carboxymethyl-(58.01)`.

In [2]:
from io import StringIO

from alphabase.spectral_library.peaks_reader import PEAKSLibraryReader

tsv_str = """m/z	z	rt (seconds)	Activation Mode	Sequence (backbone)	Modifications	Peaks Count	Peaks List	Engine
"478.77979"	"2"	"58.2"	"CID, CAD(y and b ions)"	"AAAAAAALQAK"	""	"16"	"214.11861:1.0000:b3;218.14990:0.1106:y2;285.15570:0.6207:b4;346.20847:0.1872:y3;356.19281:0.4090:b5;427.22992:0.1952:b6;459.29254:0.2122:y4;498.26703:0.1207:b7;530.32965:0.3947:y5;601.36676:0.7234:y6;611.35107:0.0564:b8;672.40387:0.9511:y7;739.40967:0.0261:b9;743.44098:0.4926:y8;814.47809:0.3936:y9;885.51520:0.0250:y10"	"DB_SEARCH"
"651.81586"	"2"	"61.2"	"CID, CAD(y and b ions)"	"AAAMTPPEEELK"	"3-Oxidation (M)-(15.99)"	"12"	"214.11861:0.0159:b3;260.19684:0.0159:y2;361.15399:0.0106:b4;389.23944:0.0106:y3;462.20169:0.0106:b5;518.28204:0.0370:y4;559.25446:0.0106:b6;647.32458:1.0000:y5;744.37738:0.0106:y6;841.43011:0.0899:y7;942.47778:0.0264:y8;1160.55029:0.0106:y10"	"DB_SEARCH"
"763.32721"	"2"	"65.5"	"CID, CAD(y and b ions)"	"AAAGEFADDPCSSVK"	"10-Carboxymethyl-(58.01)"	"20"	"214.11861:0.4500:b3;246.18120:0.0999:y2;271.14005:0.4500:b4;333.21323:0.1499:y3;400.18265:0.4498:b5;420.24524:0.1000:y4;547.25104:0.0500:b6;581.25995:0.1499:y5;618.28815:0.2998:b7;678.31268:0.7499:y6;716.28815:0.0500:b8-NH3;775.32904:0.0500:y7-H2O;776.31262:0.0500:y7-NH3;848.34204:0.1000:b9;908.36658:0.4000:y8;979.40369:1.0000:y9;1126.47205:0.9001:y10;1237.50415:0.0500:y11-H2O;1312.53613:0.3999:y12;1383.57324:0.0500:y13"	"DB_SEARCH"
"753.37427"	"3"	"92.0"	"CID, CAD(y and b ions)"	"MASNPERGEILLTELQGDSR"	"0-Acetylation (Protein N-term)-(42.01)"	"20"	"174.05833:0.0689:b1;175.11894:1.0000:y1;189.09258:0.3448:y3[2+];245.09544:0.2758:b2;262.15097:0.3793:y2;272.11520:0.0690:b5[2+];332.12744:0.3447:b3;338.17462:0.1378:y6[2+];377.17789:0.1034:y3;434.19934:0.1378:y4;443.19778:0.1034:b8[2+];446.17038:0.0689:b4;453.21976:0.2758:y8[2+];564.26117:0.1378:b10[2+];566.30383:0.1378:y10[2+];792.39038:0.1723:b14[2+];858.44977:0.2069:y15[2+];905.43225:0.1379:y8;1007.51361:0.1723:y18[2+];1018.51636:0.1379:y9"	"DB_SEARCH"
"790.07495"	"3"	"83.0"	"CID, CAD(y and b ions)"	"AAAAAAAAAPAAAATAPTTAATTAATAAQ"	""	"20"	"214.11861:0.5882:b6[2+];218.11353:0.5980:y2;285.15570:0.7745:b8[2+];289.15063:0.1078:y3;356.19281:0.7794:b5;390.19830:0.1618:y4;427.22992:1.0000:b6;440.23773:0.3137:b12[2+];475.75629:0.1667:b13[2+];498.26703:0.6912:b7;569.30414:0.7255:b8;588.29877:0.1569:y13[2+];597.31732:0.0833:b16[2+];640.34125:0.5392:b9;737.39398:0.0882:b10;780.89685:0.1520:y18[2+];808.43109:0.1078:b11;879.46820:0.1127:b12;977.48981:0.1471:y11;1175.59021:0.1961:y13"	"DB_SEARCH"
"592.80804"	"4"	"83.0"	"CID, CAD(y and b ions)"	"AAAAAAAAAPAAAATAPTTAATTAATAAQ"	""	"20"	"214.11861:1.0000:b6[2+];218.11353:0.2058:y2;231.12134:0.1697:y5[2+];285.15570:0.1516:b8[2+];289.15063:0.0542:y3;356.19281:0.2491:b5;369.20062:0.0542:b10[2+];390.19830:0.0722:y4;427.22992:0.1733:b6;489.24854:0.1263:y11[2+];498.26703:0.2130:b7;511.27484:0.0614:b14[2+];532.27252:0.0541:y6;569.30414:0.1516:b8;588.29877:0.2238:y13[2+];640.34125:0.0830:b9;737.39398:0.0469:b10;808.43109:0.0505:b11;868.45233:0.0578:b22[2+];879.46820:0.0577:b12"	"DB_SEARCH"
"""

reader = PEAKSLibraryReader()
speclib = reader.import_file(StringIO(tsv_str))

for col in ['sequence', 'charge', 'precursor_mz', 'rt', 'nAA', 'mods', 'mod_sites', 'flat_frag_start_idx', 'flat_frag_stop_idx']:
    assert col in speclib.precursor_df.columns

speclib.precursor_df

,sequence,charge,precursor_mz,rt,nAA,rt_norm,mods,mod_sites,flat_frag_start_idx,flat_frag_stop_idx
0,AAAAAAALQAK,2,478.77979,0.970000,11,0.632609,,,0,16
1,AAAMTPPEEELK,2,651.81586,1.020000,12,0.665217,Oxidation@M,4,16,28
2,AAAGEFADDPCSSVK,2,763.32721,1.091667,15,0.711957,Carboxymethyl@C,11,28,48
3,MASNPERGEILLTELQGDSR,3,753.37427,1.533333,20,1.000000,Acetyl@Protein_N-term,0,48,68
4,AAAAAAAAAPAAAATAPTTAATTAATAAQ,3,790.07495,1.383333,29,0.902174,,,68,88
5,AAAAAAAAAPAAAATAPTTAATTAATAAQ,4,592.80804,1.383333,29,0.902174,,,88,108


The same peptide observed at two charge states (`AAAAAAAAAPAAAATAPTTAATTAATAAQ`, z=3 and z=4) becomes two separate rows, each with its own fragments - not merged.

## Fragments

Fragments live in `speclib.fragment_df`, one row per ion, in the flat format used throughout AlphaBase (`mz`, `intensity`, `type`, `number`, `position`, `charge`, `loss_type` - `type` and `loss_type` are the integer codes from `alphabase.peptide.fragment.SERIES_MAPPING`/`LOSS_MAPPING`, not strings). Each precursor's own slice is `[flat_frag_start_idx, flat_frag_stop_idx)`.

In [3]:
speclib.fragment_df

,mz,intensity,type,number,position,charge,loss_type
0,214.11861,1.0000,98,3,2,1,0
1,218.14990,0.1106,121,2,8,1,0
2,285.15570,0.6207,98,4,3,1,0
3,346.20847,0.1872,121,3,7,1,0
4,356.19281,0.4090,98,5,4,1,0
...,...,...,...,...,...,...,...
103,640.34125,0.0830,98,9,8,1,0
104,737.39398,0.0469,98,10,9,1,0
105,808.43109,0.0505,98,11,10,1,0
106,868.45233,0.0578,98,22,21,2,0


In [4]:
# fragments belonging to the Carboxymethyl-modified precursor
row = speclib.precursor_df[speclib.precursor_df['sequence'] == 'AAAGEFADDPCSSVK'].iloc[0]
speclib.fragment_df.iloc[row['flat_frag_start_idx']:row['flat_frag_stop_idx']]

,mz,intensity,type,number,position,charge,loss_type
28,214.11861,0.4500,98,3,2,1,0
29,246.18120,0.0999,121,2,12,1,0
30,271.14005,0.4500,98,4,3,1,0
31,333.21323,0.1499,121,3,11,1,0
32,400.18265,0.4498,98,5,4,1,0
33,420.24524,0.1000,121,4,10,1,0
34,547.25104,0.0500,98,6,5,1,0
35,581.25995,0.1499,121,5,9,1,0
36,618.28815,0.2998,98,7,6,1,0
37,678.31268,0.7499,121,6,8,1,0


## Modifications

PEAKS' modification names are harmonized to AlphaBase/UniMod names via the `"peaks"` entry in `psm_reader.yaml`, the same `ModificationMapper`-based mechanism every other reader in `alphabase.psm_reader` uses. `mod_sites` are 1-based for side-chain modifications (PEAKS reports 0-based positions - `AlphaBase site = PEAKS position + 1`) and the fixed string `"0"` for protein N-terminal modifications.

In [5]:
speclib.precursor_df[['sequence', 'mods', 'mod_sites']]

,sequence,mods,mod_sites
0,AAAAAAALQAK,,
1,AAAMTPPEEELK,Oxidation@M,4
2,AAAGEFADDPCSSVK,Carboxymethyl@C,11
3,MASNPERGEILLTELQGDSR,Acetyl@Protein_N-term,0
4,AAAAAAAAAPAAAATAPTTAATTAATAAQ,,
5,AAAAAAAAAPAAAATAPTTAATTAATAAQ,,


Only the modifications observed in the MPIB example PEAKS library are mapped by default (`Carboxymethyl@C`, `Oxidation@M`, `Acetyl@Protein_N-term`). Extend the mapping without editing the yaml file by passing `modification_mapping` to the constructor:

```python
reader = PEAKSLibraryReader(
    modification_mapping={'Phospho@S': ['Phospho (STY)']},
)
```

A PEAKS modification name with no mapping doesn't raise - it warns and drops just that one precursor, so one unrecognized modification doesn't abort the rest of a large library:

In [6]:
import warnings

unmapped_tsv_str = """m/z	z	rt (seconds)	Activation Mode	Sequence (backbone)	Modifications	Peaks Count	Peaks List	Engine
"500.0"	"2"	"60.0"	"CID, CAD(y and b ions)"	"ANQKTVL"	"1-Phospho (STY)-(79.97)"	"2"	"214.11861:1.0000:b3;218.11353:0.5000:y2"	"DB_SEARCH"
"""

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    unmapped_speclib = PEAKSLibraryReader().import_file(StringIO(unmapped_tsv_str))

assert len(unmapped_speclib.precursor_df) == 0
print([str(w.message) for w in caught])

Unknown or mass-mismatched PEAKS modification(s) {'Phospho (STY)'} in '1-Phospho (STY)-(79.97)'. Dropping this precursor. Add a mapping via `modification_mapping` to keep it.


['Dropped 1 precursor(s) with unmapped modifications.']


Some PEAKS modification names are inherently residue-ambiguous - they name a modification once per shared delta mass, even though the underlying UniMod entries are residue-specific. `Deamidation (NQ)` is one example: `Deamidated@N` and `Deamidated@Q` are both 0.98 Da. These are resolved by checking which of the candidate residues is actually present at the reported sequence position, with the parsed mass cross-checked against the resolved modification's expected mass as a sanity guard against a wrong mapping:

In [7]:
residue_aware_tsv_str = """m/z	z	rt (seconds)	Activation Mode	Sequence (backbone)	Modifications	Peaks Count	Peaks List	Engine
"500.0"	"2"	"60.0"	"CID, CAD(y and b ions)"	"ANQKTVL"	"1-Deamidation (NQ)-(0.98)"	"2"	"214.11861:1.0000:b3;218.11353:0.5000:y2"	"DB_SEARCH"
"501.0"	"2"	"61.0"	"CID, CAD(y and b ions)"	"ANQKTVL"	"2-Deamidation (NQ)-(0.98)"	"2"	"214.11861:1.0000:b3;218.11353:0.5000:y2"	"DB_SEARCH"
"""

residue_aware_speclib = PEAKSLibraryReader().import_file(StringIO(residue_aware_tsv_str))
residue_aware_speclib.precursor_df[['sequence', 'mods', 'mod_sites']]

,sequence,mods,mod_sites
0,ANQKTVL,Deamidated@N,2
1,ANQKTVL,Deamidated@Q,3
